# Clasificacion: prediccion de inasistencias

## Objetivo

Esta libreta entrena un modelo de **clasificacion** para anticipar si una cita puede terminar como **inasistencia**.

La variable objetivo se construye como una **etiqueta de riesgo de inasistencia**:

- `Alta`: citas que historicamente terminaron como `no_show`.
- `Media`: citas que historicamente terminaron como `cancelada`.
- `Baja`: citas que historicamente fueron asistidas.

Se comparan varios modelos de clasificacion con validacion cruzada para elegir el de mejor desempeno:

- **LogisticRegression**: linea base interpretable.
- **KNeighborsClassifier**: clasifica segun casos similares.
- **DecisionTreeClassifier**: sirve como modelo base, facil de explicar.
- **RandomForestClassifier**: combina varios arboles y suele generalizar mejor.
- **GradientBoostingClassifier**: aprende corrigiendo errores de modelos anteriores.

In [1]:
from pathlib import Path
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

sns.set_theme(style="whitegrid")

SERVER_ROOT = Path.cwd()
if SERVER_ROOT.name == "notebooks":
    SERVER_ROOT = SERVER_ROOT.parents[1]
elif SERVER_ROOT.name == "ml":
    SERVER_ROOT = SERVER_ROOT.parent

ARTIFACT_DIR = SERVER_ROOT / "ml" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def load_env(path):
    env = {}
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env

env = {**load_env(SERVER_ROOT / ".env"), **os.environ}

def connect():
    conn = psycopg2.connect(
        host=env.get("DB_HOST", "localhost"),
        port=int(env.get("DB_PORT", 5432)),
        user=env.get("DB_USER", "postgres"),
        password=env.get("DB_PASSWORD", ""),
        dbname=env.get("DB_NAME", "db_barberia"),
    )
    with conn.cursor() as cur:
        cur.execute("SET search_path TO core, catalogo, admin, public")
    return conn

conn = connect()
df = pd.read_sql_query("SELECT * FROM analitica.ml_citas_dataset ORDER BY id", conn)
df["fecha"] = pd.to_datetime(df["fecha"])
df["hora"] = pd.to_datetime(df["hora"].astype(str), format="%H:%M:%S", errors="coerce").dt.hour
df.head()

C:\Users\Javi\AppData\Local\Temp\ipykernel_5192\267030489.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM analitica.ml_citas_dataset ORDER BY id", conn)


,id,cliente_ref,cliente_nombre,local_id,local_nombre,servicio_id,servicio_nombre,barbero_id,barbero_nombre,fecha,...,monto_pagado,estado_cita,recordatorio_enviado,frecuencia_cliente,recencia_dias,gasto_total_cliente,no_show_rate_cliente,canal,seed_run_id,created_at
0,1,20,Eduardo Reyes,1,Barbería Carlyn,12,Paquete 2 - Ritual Caballero,35,Luis Hernández,2026-05-22,...,319.15,asistio,True,3,12,889.09,0.0794,walk-in,20260722084715,2026-07-22 08:47:15.164613
1,2,204,Emiliano Cruz,1,Barbería Carlyn,11,Paquete 1 - Corte First Class,34,Carlos Méndez,2026-06-16,...,246.26,asistio,False,4,58,584.85,0.0876,telefono,20260722084715,2026-07-22 08:47:15.164613
2,3,159,Pablo Lopez,1,Barbería Carlyn,12,Paquete 2 - Ritual Caballero,34,Carlos Méndez,2026-07-16,...,313.55,asistio,True,8,23,1075.99,0.0815,web,20260722084715,2026-07-22 08:47:15.164613
3,4,106,Sergio Aguilar,1,Barbería Carlyn,12,Paquete 2 - Ritual Caballero,34,Carlos Méndez,2026-07-13,...,315.20,asistio,False,2,66,660.94,0.2215,web,20260722084715,2026-07-22 08:47:15.164613
4,5,118,Miguel Moreno,1,Barbería Carlyn,13,Paquete 3 - Premium Black,34,Carlos Méndez,2026-02-19,...,376.15,asistio,True,3,58,429.97,0.3574,walk-in,20260722084715,2026-07-22 08:47:15.164613


## 1. Revision inicial del dataset

Primero verificamos cuantas citas hay por estado. Las citas `pendiente` no se usan para entrenar porque aun no sabemos su resultado real.

In [2]:
df["estado_cita"].value_counts().rename_axis("estado").reset_index(name="total")

,estado,total
0,asistio,744
1,no_show,106
2,pendiente,97
3,cancelada,53


## 2. Construccion de la variable objetivo

Quitamos citas pendientes y creamos la columna `riesgo_inasistencia`. Esta columna es lo que el modelo intentara aprender.

El problema se maneja como clasificacion multiclase:

| Estado historico | Etiqueta aprendida |
|---|---|
| no_show | Alta |
| cancelada | Media |
| asistio | Baja |

In [ ]:
data = df[df["estado_cita"] != "pendiente"].copy()

label_map = {
    "no_show": "Alta",
    "cancelada": "Media",
    "asistio": "Baja",
}

data["riesgo_inasistencia"] = data["estado_cita"].map(label_map)
data = data.dropna(subset=["riesgo_inasistencia"]).copy()

data[["estado_cita", "riesgo_inasistencia"]].value_counts().reset_index(name="total")

## 3. Variables predictoras

Usamos variables que existirian antes de que ocurra la cita:

- sucursal, servicio y barbero
- dia, semana y hora
- precio y duracion del servicio
- si se envio recordatorio
- comportamiento historico del cliente: frecuencia, recencia, gasto y tasa previa de no-show
- canal por el que se genero la cita

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

features = [
    "local_nombre", "servicio_nombre", "barbero_nombre",
    "dia_semana", "semana", "hora", "precio", "duracion",
    "recordatorio_enviado", "frecuencia_cliente", "recencia_dias",
    "gasto_total_cliente", "no_show_rate_cliente", "canal",
]

categorical = ["local_nombre", "servicio_nombre", "barbero_nombre", "canal"]
numeric = [column for column in features if column not in categorical]

X = data[features]
y = data["riesgo_inasistencia"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
])

## 4. Comparacion con validacion cruzada

Entrenamos varios clasificadores y los comparamos con validacion cruzada. La metrica principal es `f1_weighted`, porque toma en cuenta el desbalance entre etiquetas.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1200, class_weight="balanced"),
    "KNeighborsClassifier": KNeighborsClassifier(n_neighbors=9, weights="distance"),
    "DecisionTreeClassifier": DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=42),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=160, max_depth=9, min_samples_leaf=6, class_weight="balanced", random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=120, learning_rate=0.06, max_depth=3, random_state=42),
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
}

comparison = []
for name, estimator in models.items():
    pipeline = Pipeline([("prep", preprocess), ("model", estimator)])
    scores = cross_validate(pipeline, X_train, y_train, cv=5, scoring=scoring, n_jobs=None)
    comparison.append({
        "modelo": name,
        "accuracy_cv": scores["test_accuracy"].mean(),
        "precision_cv": scores["test_precision"].mean(),
        "recall_cv": scores["test_recall"].mean(),
        "f1_cv": scores["test_f1"].mean(),
    })

comparison_df = pd.DataFrame(comparison).sort_values("f1_cv", ascending=False)
comparison_df

## 5. Entrenamiento del mejor clasificador

Elegimos el modelo con mejor `f1_cv`, lo entrenamos con el conjunto de entrenamiento y lo evaluamos con datos de prueba.

In [ ]:
best_name = comparison_df.iloc[0]["modelo"]
best_model = Pipeline([("prep", preprocess), ("model", models[best_name])])
best_model.fit(X_train, y_train)
test_pred = best_model.predict(X_test)
test_confidence = best_model.predict_proba(X_test).max(axis=1)

test_metrics = {
    "modelo": best_name,
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, average="weighted", zero_division=0),
    "recall": recall_score(y_test, test_pred, average="weighted", zero_division=0),
    "f1": f1_score(y_test, test_pred, average="weighted", zero_division=0),
}

pd.DataFrame([test_metrics])

## 6. Matriz de confusion

La matriz ayuda a ver donde se equivoca el modelo:

- Alta.
- Media.
- Baja.

In [ ]:
labels = ["Alta", "Media", "Baja"]
cm = confusion_matrix(y_test, test_pred, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title(f"Matriz de confusion - {best_name}")
plt.show()

print(classification_report(y_test, test_pred, labels=labels, zero_division=0))

## 7. Clasificacion de citas pendientes

Aplicamos el modelo a citas futuras o pendientes para obtener directamente una etiqueta de seguimiento.

In [ ]:
candidates = df[df["estado_cita"] == "pendiente"].copy()
if candidates.empty:
    candidates = df.sort_values("fecha", ascending=False).head(80).copy()

candidates["riesgo_inasistencia"] = best_model.predict(candidates[features])
candidates["confianza_modelo"] = best_model.predict_proba(candidates[features]).max(axis=1)
prioridad = {"Alta": 3, "Media": 2, "Baja": 1}
candidates["prioridad_orden"] = candidates["riesgo_inasistencia"].map(prioridad).fillna(0)
top_risk = candidates.sort_values(["prioridad_orden", "confianza_modelo", "fecha", "hora"], ascending=[False, False, True, True]).head(15)

top_risk[["fecha", "hora", "cliente_nombre", "servicio_nombre", "local_nombre", "riesgo_inasistencia"]]

## 8. Guardado del modelo

Guardamos el modelo entrenado en formato `.joblib`. Este archivo se puede cargar despues para predecir sin volver a entrenar.

In [ ]:
artifact_path = ARTIFACT_DIR / "notebook_clasificacion_no_asistencia.joblib"
joblib.dump(best_model, artifact_path)
artifact_path